# CA2 Mini Project: Customer Segmentation for an Online Retail Business
**DMBI · Continuous Assessment 2** · Soniya Lulla (69) · Sejal Alle (64) · Sahil Setya (48) · Ronit Ahuja (63)

**Pipeline:** Problem → Data → Preprocessing → Algorithm → Results → Evaluation → Insights → Decision

| Item | Choice |
|---|---|
| Dataset | UCI Online Retail (541,909 transactions, 01/12/2010 to 09/12/2011, UK-based non-store retailer) |
| Features | RFM: Recency, Frequency, Monetary |
| Main algorithm | K-Means (compared with Gaussian Mixture and Ward hierarchical) |
| Evaluation | Silhouette, Davies-Bouldin, Calinski-Harabasz, seed stability (ARI) |
| GUI | `ipywidgets` dashboard in Section 10 (country filter, k slider, customer lookup, what-if tool) |

**How to run in Colab:** upload this file (File → Upload notebook), then *Runtime → Run all*. The dataset downloads itself from UCI (about 20 MB, may take a minute). If the download fails, the notebook asks you to upload `Online Retail.xlsx` or the zip.

## 1. Setup

In [ ]:
import importlib, subprocess, sys, os, io, zipfile, time, urllib.request, warnings
for pkg in ["openpyxl"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold",
                     "axes.titlesize": 12})

RESULTS_DIR = "Results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RFM_COLS = ["Recency", "Frequency", "Monetary"]

# Colour-blind-safe palette, one fixed colour per segment name
SEG_COLORS = {"Champions": "#0072B2", "Loyal Customers": "#009E73",
              "Potential Loyalists": "#E69F00", "At Risk": "#D55E00",
              "Hibernating": "#8C8C8C", "Lost Customers": "#CC79A7"}
print("Setup complete.")

## 2. Load the dataset
Source: [UCI Machine Learning Repository, Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail). Eight fields: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
XLSX = "Online Retail.xlsx"
CACHE = "online_retail_raw.pkl"

def _extract_zip(data: bytes):
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        name = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
        with open(XLSX, "wb") as f:
            f.write(z.read(name))

def get_dataset_file():
    if os.path.exists(XLSX) or os.path.exists(CACHE):
        return
    for attempt in range(1, 4):
        try:
            print(f"Downloading from UCI (attempt {attempt}/3)...")
            with urllib.request.urlopen(DATA_URL, timeout=180) as r:
                _extract_zip(r.read())
            print("Download complete.")
            return
        except Exception as e:
            print("  failed:", e)
            time.sleep(3)
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit(f"Download failed. Place '{XLSX}' next to this notebook and rerun.")
    print("Automatic download failed. Please upload 'Online Retail.xlsx' (or the UCI zip).")
    for name, data in files.upload().items():
        if name.lower().endswith(".zip"):
            _extract_zip(data)
        elif name.lower().endswith(".xlsx"):
            with open(XLSX, "wb") as f:
                f.write(data)

get_dataset_file()
if os.path.exists(CACHE):
    raw = pd.read_pickle(CACHE)
else:
    print("Reading Excel file (about a minute)...")
    raw = pd.read_excel(XLSX, dtype={"InvoiceNo": str, "StockCode": str})
    raw.to_pickle(CACHE)

print(f"Loaded {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Period: {raw.InvoiceDate.min():%d %b %Y} to {raw.InvoiceDate.max():%d %b %Y}")
raw.head()

## 3. Data audit and preprocessing
Cleaning rules (each one is justified because it would distort customer behaviour):
1. Drop rows with no `CustomerID` (cannot be assigned to a customer).
2. Drop exact duplicate rows.
3. Drop cancellations (invoice numbers starting with `C`) and non-positive `Quantity` or `UnitPrice`.
4. Drop non-product codes (POST, M, BANK CHARGES, DOT and similar) because they are fees, not purchases.
5. Create `TotalAmount = Quantity × UnitPrice`.

In [ ]:
audit = {
    "Rows (raw)": len(raw),
    "Missing CustomerID": int(raw.CustomerID.isna().sum()),
    "Missing Description": int(raw.Description.isna().sum()),
    "Duplicate rows": int(raw.duplicated().sum()),
    "Cancellation invoices (start with C)": int(raw.InvoiceNo.astype(str).str.startswith("C").sum()),
    "Quantity <= 0": int((raw.Quantity <= 0).sum()),
    "UnitPrice <= 0": int((raw.UnitPrice <= 0).sum()),
}
audit_df = pd.DataFrame({"Count": audit}); audit_df["% of rows"] = audit_df.Count / len(raw) * 100
display(audit_df)

df = raw.copy()
steps = [("Raw rows", len(df))]
df = df.dropna(subset=["CustomerID"]);                          steps.append(("After dropping missing CustomerID", len(df)))
df = df.drop_duplicates();                                       steps.append(("After dropping duplicates", len(df)))
df = df[~df.InvoiceNo.astype(str).str.startswith("C")];          steps.append(("After dropping cancellations", len(df)))
df = df[(df.Quantity > 0) & (df.UnitPrice > 0)];                 steps.append(("After dropping invalid Quantity/UnitPrice", len(df)))
df = df[df.StockCode.astype(str).str.match(r"^\d{5}")];          steps.append(("After dropping non-product codes", len(df)))
df["CustomerID"] = df.CustomerID.astype(int)
df["TotalAmount"] = df.Quantity * df.UnitPrice
df = df.reset_index(drop=True)

flow = pd.DataFrame(steps, columns=["Step", "Rows"]); flow["Removed"] = (-flow.Rows.diff()).fillna(0).astype(int)
display(flow)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(flow.Step[::-1], flow.Rows[::-1], color="#0072B2")
for y, v in enumerate(flow.Rows[::-1]):
    ax.text(v, y, f" {v:,}", va="center", fontsize=9)
ax.set_title("Rows remaining after each cleaning step"); ax.set_xlim(0, flow.Rows.max() * 1.15); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
print(f"Clean dataset: {len(df):,} rows, {df.CustomerID.nunique():,} customers, {df.InvoiceNo.nunique():,} invoices")

## 4. Exploratory data analysis
Baseline measures before any segmentation.

In [ ]:
kpi = {
    "Total revenue": f"£{df.TotalAmount.sum():,.0f}",
    "Customers": f"{df.CustomerID.nunique():,}",
    "Invoices": f"{df.InvoiceNo.nunique():,}",
    "Avg order value": f"£{df.groupby('InvoiceNo').TotalAmount.sum().mean():,.2f}",
    "Repeat customers": f"{(df.groupby('CustomerID').InvoiceNo.nunique() > 1).mean() * 100:.1f}%",
    "Countries": f"{df.Country.nunique()}",
}
display(pd.DataFrame(kpi, index=["Value"]).T)

fig, axs = plt.subplots(1, 3, figsize=(20, 5.2))

top_c = df.groupby("Country").TotalAmount.sum().nlargest(10).sort_values()
axs[0].barh(top_c.index, top_c.values / 1e3, color="#0072B2")
axs[0].set_title("Top 10 countries by revenue"); axs[0].set_xlabel("Revenue (£ thousand)"); axs[0].grid(axis="y", visible=False)

monthly = df.set_index("InvoiceDate").TotalAmount.resample("MS").sum()
axs[1].plot(monthly.index, monthly.values / 1e3, marker="o", color="#009E73")
axs[1].set_title("Monthly revenue"); axs[1].set_ylabel("Revenue (£ thousand)")
axs[1].annotate("Dec 2011 is a partial month\n(data ends 9 Dec)", xy=(monthly.index[-1], monthly.values[-1] / 1e3),
                xytext=(-130, 40), textcoords="offset points", fontsize=9, arrowprops=dict(arrowstyle="->"))
axs[1].tick_params(axis="x", rotation=35)

desc = df.groupby("StockCode").Description.agg(lambda s: s.mode().iat[0] if not s.mode().empty else "")
top_p = df.groupby("StockCode").Quantity.sum().nlargest(10).sort_values()
labels = [desc[c][:28] for c in top_p.index]
axs[2].barh(labels, top_p.values / 1e3, color="#E69F00")
axs[2].set_title("Top 10 products by quantity sold"); axs[2].set_xlabel("Units (thousand)"); axs[2].grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

# Purchase frequency
freq = df.groupby("CustomerID").InvoiceNo.nunique()
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(freq.clip(upper=30), bins=30, color="#0072B2")
ax.set_title("Invoices per customer (clipped at 30)"); ax.set_xlabel("Invoices"); ax.set_ylabel("Customers")
plt.tight_layout(); plt.show()

## 5. RFM feature engineering
* **Recency**: days between the reference date (day after the last transaction) and the customer's last purchase.
* **Frequency**: number of unique invoices.
* **Monetary**: total spend.

Each is also scored 1 to 5 by quantiles (5 is best) to give an interpretable, rule-based segmentation that we compare with K-Means later.

In [ ]:
REF_DATE = df.InvoiceDate.max() + pd.Timedelta(days=1)

def build_rfm(d):
    g = d.groupby("CustomerID")
    return pd.DataFrame({
        "Recency": (REF_DATE - g.InvoiceDate.max()).dt.days,
        "Frequency": g.InvoiceNo.nunique(),
        "Monetary": g.TotalAmount.sum(),
    })

SEG_MAP = {r"[1-2][1-2]": "Hibernating", r"[1-2][3-4]": "At Risk", r"[1-2]5": "Can't Lose Them",
           r"3[1-2]": "About to Sleep", r"33": "Need Attention", r"[3-4][4-5]": "Loyal Customers",
           r"41": "Promising", r"51": "New Customers", r"[4-5][2-3]": "Potential Loyalists", r"5[4-5]": "Champions"}

def score_rfm(rfm):
    s = rfm.copy()
    s["R"] = pd.qcut(s.Recency.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
    s["F"] = pd.qcut(s.Frequency.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    s["M"] = pd.qcut(s.Monetary.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    s["RFM_Score"] = s.R.astype(str) + s.F.astype(str) + s.M.astype(str)
    s["RuleSegment"] = (s.R.astype(str) + s.F.astype(str)).replace(SEG_MAP, regex=True)
    return s

rfm = build_rfm(df)
rfm_scored = score_rfm(rfm)
print(f"{len(rfm):,} customers | reference date {REF_DATE:%d %b %Y}")
display(rfm.describe().T)

fig, axs = plt.subplots(2, 3, figsize=(18, 7))
for i, c in enumerate(RFM_COLS):
    axs[0, i].hist(rfm[c], bins=50, color="#0072B2"); axs[0, i].set_title(f"{c} (raw, right-skewed)")
    axs[1, i].hist(np.log1p(rfm[c]), bins=50, color="#009E73"); axs[1, i].set_title(f"{c} after log(1+x)")
plt.tight_layout(); plt.show()
print("Skewness raw -> log:", {c: (round(rfm[c].skew(), 2), round(np.log1p(rfm[c]).skew(), 2)) for c in RFM_COLS})

## 6. Algorithm selection and evaluation
Features are log-transformed (reduces skew) and standardised so each dimension weighs equally in the distance. We sweep k, then compare three algorithms and test stability.

In [ ]:
Z_all = StandardScaler().fit_transform(np.log1p(rfm[RFM_COLS]))

rows = []
for k in range(2, 11):
    km = KMeans(k, n_init=10, random_state=SEED).fit(Z_all)
    rows.append({"k": k, "Inertia": km.inertia_,
                 "Silhouette": silhouette_score(Z_all, km.labels_),
                 "Davies-Bouldin": davies_bouldin_score(Z_all, km.labels_),
                 "Calinski-Harabasz": calinski_harabasz_score(Z_all, km.labels_)})
ksel = pd.DataFrame(rows).set_index("k")
display(ksel)

fig, axs = plt.subplots(1, 3, figsize=(18, 4.3))
for ax, col, title in zip(axs, ["Inertia", "Silhouette", "Davies-Bouldin"],
                          ["Elbow (inertia, lower is better)", "Silhouette (higher is better)", "Davies-Bouldin (lower is better)"]):
    ax.plot(ksel.index, ksel[col], marker="o", color="#222222"); ax.axvline(4, ls="--", color="#D55E00")
    ax.set_title(title); ax.set_xlabel("k")
axs[0].text(4.1, ksel.Inertia.max() * 0.9, "k = 4", color="#D55E00")
plt.tight_layout(); plt.show()
best_sil = ksel.Silhouette.idxmax()
print(f"Highest silhouette is at k = {best_sil}. k = 4 is chosen as a balance of statistical quality and "
      f"the number of marketing actions a team can manage; the GUI in Section 10 lets you test other k values.")

In [ ]:
# Algorithm comparison at k = 4 on the same data
algos = {
    "K-Means": lambda: KMeans(4, n_init=10, random_state=SEED),
    "Gaussian Mixture": lambda: GaussianMixture(4, n_init=3, random_state=SEED),
    "Ward hierarchical": lambda: AgglomerativeClustering(4, linkage="ward"),
}
lab, rows = {}, []
for name, make in algos.items():
    t0 = time.perf_counter(); lab[name] = make().fit_predict(Z_all); dt = time.perf_counter() - t0
    rows.append({"Algorithm": name, "Silhouette": silhouette_score(Z_all, lab[name]),
                 "Davies-Bouldin": davies_bouldin_score(Z_all, lab[name]),
                 "Calinski-Harabasz": calinski_harabasz_score(Z_all, lab[name]), "Seconds": dt})
comp = pd.DataFrame(rows).set_index("Algorithm")
comp["ARI vs K-Means"] = [adjusted_rand_score(lab["K-Means"], lab[a]) for a in comp.index]
display(comp)

# Stability: 10 different seeds vs seed 42
base = KMeans(4, n_init=10, random_state=SEED).fit_predict(Z_all)
aris = [adjusted_rand_score(base, KMeans(4, n_init=10, random_state=s).fit_predict(Z_all)) for s in range(1, 11)]
print(f"Seed stability: mean ARI = {np.mean(aris):.4f} (min {np.min(aris):.4f}) over 10 seeds")

fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, col, better in zip(axs, ["Silhouette", "Davies-Bouldin", "Calinski-Harabasz"], ["higher", "lower", "higher"]):
    ax.bar(comp.index, comp[col], color=["#0072B2", "#E69F00", "#009E73"]); ax.set_title(f"{col} ({better} is better)")
    ax.tick_params(axis="x", rotation=15); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 7. Final segmentation (K-Means) and profiles
Segment names are assigned by a transparent rule: each cluster gets a *value score* = standardised Frequency + standardised Monetary − standardised Recency (lower recency is better). Clusters are ranked by that score and given names from best to worst. The names are labels for decision-making, not facts about customers' attitudes.

In [ ]:
NAMES_BY_K = {
    2: ["Champions", "Hibernating"],
    3: ["Champions", "Potential Loyalists", "Hibernating"],
    4: ["Champions", "Loyal Customers", "At Risk", "Hibernating"],
    5: ["Champions", "Loyal Customers", "Potential Loyalists", "At Risk", "Hibernating"],
    6: ["Champions", "Loyal Customers", "Potential Loyalists", "At Risk", "Hibernating", "Lost Customers"],
}
ACTIONS = {
    "Champions": "Protect value: VIP perks, early access, referral and loyalty rewards.",
    "Loyal Customers": "Keep engaged: loyalty tiers, cross-sell, personalised offers.",
    "Potential Loyalists": "Build the habit: membership offer, second/third-purchase incentive, onboarding.",
    "At Risk": "Recover value: win-back email with a personalised discount, re-engagement campaign.",
    "Hibernating": "Low-cost reactivation: light win-back email, avoid heavy discounting.",
    "Lost Customers": "Last-chance survey or suppress from paid campaigns.",
}

def fit_segments(rfm_df, k=4, algo="K-Means", seed=SEED):
    X = np.log1p(rfm_df[RFM_COLS]); scaler = StandardScaler().fit(X); Z = scaler.transform(X)
    if algo == "K-Means":
        labels = KMeans(k, n_init=10, random_state=seed).fit_predict(Z)
    elif algo == "Gaussian Mixture":
        labels = GaussianMixture(k, n_init=3, random_state=seed).fit_predict(Z)
    else:
        labels = AgglomerativeClustering(k, linkage="ward").fit_predict(Z)
    centers = pd.DataFrame(Z, columns=RFM_COLS).groupby(labels).mean()
    value = -centers.Recency + centers.Frequency + centers.Monetary
    order = value.sort_values(ascending=False).index.tolist()
    name_map = {c: NAMES_BY_K[k][i] for i, c in enumerate(order)}
    seg = pd.Series(labels, index=rfm_df.index).map(name_map).rename("Segment")
    try:
        metrics = {"Silhouette": silhouette_score(Z, labels), "Davies-Bouldin": davies_bouldin_score(Z, labels),
                   "Calinski-Harabasz": calinski_harabasz_score(Z, labels)}
    except ValueError:
        metrics = {}
    return {"seg": seg, "labels": labels, "Z": Z, "scaler": scaler, "centers": centers, "name_map": name_map,
            "names_ordered": NAMES_BY_K[k], "k": k, "algo": algo, "metrics": metrics}

def segment_profile(rfm_df, res):
    d = rfm_df.join(res["seg"])
    g = d.groupby("Segment")
    p = pd.DataFrame({"Customers": g.size(), "Customer share %": g.size() / len(d) * 100,
                      "Median Recency (days)": g.Recency.median(), "Median Frequency": g.Frequency.median(),
                      "Median Monetary (£)": g.Monetary.median(), "Avg Monetary (£)": g.Monetary.mean(),
                      "Revenue (£)": g.Monetary.sum(), "Revenue share %": g.Monetary.sum() / d.Monetary.sum() * 100})
    return p.reindex(res["names_ordered"]).dropna(how="all")

res = fit_segments(rfm, k=4)
rfm_seg = rfm.join(res["seg"])
profile = segment_profile(rfm, res)
display(profile)
print({k: round(v, 4) for k, v in res["metrics"].items()})

In [ ]:
def plot_overview(rfm_df, res, title=None):
    prof = segment_profile(rfm_df, res); order = prof.index.tolist()
    colors = [SEG_COLORS.get(n, "#888888") for n in order]
    d = rfm_df.join(res["seg"])
    fig = plt.figure(figsize=(18, 10)); gs = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.28)

    # (a) customer share vs revenue share
    ax = fig.add_subplot(gs[0, 0]); y = np.arange(len(order)); h = 0.38
    ax.barh(y - h / 2, prof["Customer share %"], h, label="Customers", color="#9AA5B1")
    ax.barh(y + h / 2, prof["Revenue share %"], h, label="Revenue", color=colors)
    for i, (a, b) in enumerate(zip(prof["Customer share %"], prof["Revenue share %"])):
        ax.text(a, i - h / 2, f" {a:.0f}%", va="center", fontsize=8); ax.text(b, i + h / 2, f" {b:.0f}%", va="center", fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.legend(loc="lower right")
    ax.set_title("Share of customers vs share of revenue"); ax.set_xlabel("%"); ax.grid(axis="y", visible=False)

    # (b) heatmap of standardised centres, annotated with medians
    ax = fig.add_subplot(gs[0, 1]); C = res["centers"].rename(index=res["name_map"]).reindex(order)
    im = ax.imshow(C.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    med = prof[["Median Recency (days)", "Median Frequency", "Median Monetary (£)"]].values
    for i in range(len(order)):
        for j in range(3):
            ax.text(j, i, f"{med[i, j]:,.0f}", ha="center", va="center", fontsize=10, fontweight="bold")
    ax.set_xticks(range(3)); ax.set_xticklabels(["Recency\n(days)", "Frequency\n(invoices)", "Monetary\n(£)"])
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.grid(False)
    ax.set_title("Segment medians (colour = standardised level)"); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

    # (c) PCA projection
    ax = fig.add_subplot(gs[0, 2]); pca = PCA(2, random_state=SEED).fit(res["Z"]); P = pca.transform(res["Z"])
    idx = np.random.RandomState(SEED).permutation(len(P))[:4000]
    for n in order:
        m = (res["seg"].values[idx] == n); ax.scatter(P[idx][m, 0], P[idx][m, 1], s=8, alpha=0.55, color=SEG_COLORS.get(n, "#888"), label=n)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% variance)"); ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
    ax.set_title("Customers in PCA space"); ax.legend(markerscale=2, fontsize=8)

    # (d-f) distributions per segment
    for j, (c, lab) in enumerate(zip(RFM_COLS, ["Recency (days)", "Frequency (invoices)", "Monetary (£)"])):
        ax = fig.add_subplot(gs[1, j])
        data = [d.loc[d.Segment == n, c].values for n in order]
        bp = ax.boxplot(data, labels=order, patch_artist=True, showfliers=False)
        for patch, col in zip(bp["boxes"], colors): patch.set_facecolor(col); patch.set_alpha(0.8)
        for med_line in bp["medians"]: med_line.set_color("black")
        ax.set_title(f"{lab} by segment"); ax.tick_params(axis="x", rotation=25); ax.grid(axis="x", visible=False)
        if c != "Recency": ax.set_yscale("log")
    fig.suptitle(title or f"{res['algo']} segmentation, k = {res['k']}", fontsize=15, fontweight="bold", y=0.98)
    plt.show()

plot_overview(rfm, res)

In [ ]:
# Revenue concentration (Pareto) and donut of segment sizes
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
order = profile.index.tolist()
axs[0].pie(profile.Customers, labels=order, colors=[SEG_COLORS[n] for n in order], autopct="%1.0f%%",
           startangle=90, wedgeprops=dict(width=0.42, edgecolor="white"), pctdistance=0.79)
axs[0].set_title("Customer mix by segment")

s = rfm.Monetary.sort_values(ascending=False).reset_index(drop=True)
x = (np.arange(1, len(s) + 1) / len(s)) * 100; cum = s.cumsum() / s.sum() * 100
axs[1].plot(x, cum, color="#0072B2", lw=2); axs[1].plot([0, 100], [0, 100], "--", color="#999")
top20 = cum[int(len(s) * 0.2) - 1]
axs[1].scatter([20], [top20], color="#D55E00", zorder=5); axs[1].annotate(f"Top 20% of customers\n= {top20:.0f}% of revenue", (20, top20), xytext=(35, top20 - 22), arrowprops=dict(arrowstyle="->"))
axs[1].set_title("Revenue concentration (cumulative)"); axs[1].set_xlabel("% of customers (highest spenders first)"); axs[1].set_ylabel("% of revenue")
plt.tight_layout(); plt.show()

## 8. Comparison with rule-based RFM scores
The quantile-score segments (Champions, At Risk, Hibernating and so on) are the textbook RFM approach. This table shows how K-Means groups relate to them.

In [ ]:
cross = pd.crosstab(rfm_scored.RuleSegment, rfm_seg.Segment)
cross = cross[[n for n in res["names_ordered"] if n in cross.columns]]
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
fig, axs = plt.subplots(1, 2, figsize=(17, 5.5))

rc = rfm_scored.RuleSegment.value_counts().sort_values()
axs[0].barh(rc.index, rc.values, color="#0072B2"); axs[0].set_title("Rule-based RFM segments (quantile scores)"); axs[0].grid(axis="y", visible=False)
for i, v in enumerate(rc.values): axs[0].text(v, i, f" {v:,}", va="center", fontsize=8)

im = axs[1].imshow(cross_pct.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
for i in range(cross_pct.shape[0]):
    for j in range(cross_pct.shape[1]):
        v = cross_pct.values[i, j]; axs[1].text(j, i, f"{v:.0f}%", ha="center", va="center", color="white" if v > 55 else "black", fontsize=9)
axs[1].set_xticks(range(cross_pct.shape[1])); axs[1].set_xticklabels(cross_pct.columns, rotation=20)
axs[1].set_yticks(range(cross_pct.shape[0])); axs[1].set_yticklabels(cross_pct.index); axs[1].grid(False)
axs[1].set_title("Where each rule-based segment lands in K-Means (row %)")
plt.tight_layout(); plt.show()
print("Adjusted Rand Index between the two segmentations is low by design: one cuts each dimension into quintiles, the other finds natural groups in log-space.")

## 9. Insights and business decision
Auto-generated from the results above, followed by the recommended action for each segment.

In [ ]:
def insights(rfm_df, res):
    p = segment_profile(rfm_df, res); out = []
    top = p["Revenue share %"].idxmax()
    out.append(f"{top} generate {p.loc[top, 'Revenue share %']:.0f}% of revenue from {p.loc[top, 'Customer share %']:.0f}% of customers.")
    best = res["names_ordered"][0]
    if best in p.index:
        out.append(f"{best} ({int(p.loc[best, 'Customers']):,} customers) last ordered a median of {p.loc[best, 'Median Recency (days)']:.0f} days ago, "
                   f"with {p.loc[best, 'Median Frequency']:.0f} invoices and a median spend of £{p.loc[best, 'Median Monetary (£)']:,.0f}: protect them.")
    worst = res["names_ordered"][-1]
    if worst in p.index:
        out.append(f"{worst} are {p.loc[worst, 'Customer share %']:.0f}% of customers but only {p.loc[worst, 'Revenue share %']:.1f}% of revenue: use low-cost channels, not discounts.")
    sil = res["metrics"].get("Silhouette")
    if sil is not None:
        out.append(f"Silhouette = {sil:.2f}: " + ("clear separation." if sil >= 0.5 else "moderate separation; segments are useful bands of a continuous customer spread, not sharply isolated groups."))
    return out

for line in insights(rfm, res): print("•", line)
display(pd.DataFrame({"Segment": profile.index, "Customers": profile.Customers.astype(int).values,
                      "Revenue share %": profile["Revenue share %"].round(1).values,
                      "Recommended action": [ACTIONS[n] for n in profile.index]}).set_index("Segment"))

**Limits to state in the viva:** RFM describes past behaviour (correlation, not causation); the data is one retailer over 12 months; revenue is not profit; and no campaign has been run, so uplift of any action is untested. That last point is exactly what our CA3 topic, uplift modelling, addresses.

## 10. Interactive GUI
Run the next cell to get a dashboard with four tabs:
1. **Segmentation**: choose a country, algorithm and number of segments, then press *Run segmentation* to refresh KPIs, tables and all charts.
2. **Customer lookup**: type a CustomerID to see its RFM values, segment and recommended action.
3. **What-if**: slide Recency, Frequency and Monetary to see which segment a new customer would fall into and how confident the assignment is.
4. **Export**: save all tables and charts data to CSV/ZIP.

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output, HTML

STATE = {}
L = W.Layout
countries = ["All countries"] + sorted(df.Country.unique())

w_country = W.Dropdown(options=countries, value="All countries", description="Country", layout=L(width="300px"))
w_algo = W.Dropdown(options=["K-Means", "Gaussian Mixture", "Ward hierarchical"], value="K-Means", description="Algorithm", layout=L(width="300px"))
w_k = W.IntSlider(value=4, min=2, max=6, description="Segments k", continuous_update=False, layout=L(width="300px"))
w_run = W.Button(description="Run segmentation", button_style="primary", icon="play", layout=L(width="190px"))
out_seg = W.Output()

def card(label, value):
    return (f"<div style='display:inline-block;min-width:150px;margin:4px 8px 4px 0;padding:10px 14px;border:1px solid #bbb;"
            f"border-radius:8px;background:#f6f6f4;color:#111'><div style='font-size:11px;text-transform:uppercase;letter-spacing:.05em;color:#555'>{label}</div>"
            f"<div style='font-size:20px;font-weight:700'>{value}</div></div>")

def run_segmentation(_=None):
    with out_seg:
        clear_output(wait=True)
        country, algo, k = w_country.value, w_algo.value, w_k.value
        d = df if country == "All countries" else df[df.Country == country]
        r = build_rfm(d)
        if len(r) < max(30, k * 10):
            print(f"Only {len(r)} customers in {country}: too few to form {k} segments. Pick another country or a smaller k."); return
        w_run.disabled = True
        try:
            rs = fit_segments(r, k, algo)
        finally:
            w_run.disabled = False
        STATE.update(rfm=r, res=rs, country=country)
        cust_range = f"{len(r):,}"
        html = (card("Customers", cust_range) + card("Revenue", f"£{r.Monetary.sum():,.0f}") +
                card("Avg spend / customer", f"£{r.Monetary.mean():,.0f}") + card("Algorithm", f"{algo}, k={k}") +
                card("Silhouette", f"{rs['metrics'].get('Silhouette', float('nan')):.3f}") +
                card("Davies-Bouldin", f"{rs['metrics'].get('Davies-Bouldin', float('nan')):.3f}"))
        display(HTML(f"<h3 style='margin:6px 0'>{country}</h3>" + html))
        for line in insights(r, rs): print("•", line)
        display(segment_profile(r, rs).round(1))
        plot_overview(r, rs, f"{country}: {algo}, k = {k}")
        display(pd.DataFrame({"Recommended action": [ACTIONS[n] for n in rs["names_ordered"] if n in set(rs['seg'])]},
                             index=[n for n in rs["names_ordered"] if n in set(rs['seg'])]))
w_run.on_click(run_segmentation)
tab1 = W.VBox([W.HBox([W.VBox([w_country, w_algo, w_k]), w_run], layout=L(align_items="center")), out_seg])

# ---- Tab 2: customer lookup ----
w_id = W.IntText(value=int(rfm.index[0]), description="CustomerID", layout=L(width="260px"))
w_find = W.Button(description="Look up", icon="search", button_style="info")
w_rand = W.Button(description="Random customer", icon="random")
out_find = W.Output()

def assign(rfm_values, res):
    """Nearest-centre assignment for raw (Recency, Frequency, Monetary); returns segment, runner-up, ratio."""
    z = res["scaler"].transform(np.log1p(pd.DataFrame([rfm_values], columns=RFM_COLS)))[0]
    cen = res["centers"].values; dist = np.linalg.norm(cen - z, axis=1); order = np.argsort(dist)
    nm = res["name_map"]; cid = res["centers"].index
    return nm[cid[order[0]]], nm[cid[order[1]]], dist[order[1]] / max(dist[order[0]], 1e-9), dist[order[0]]

def lookup(_=None):
    with out_find:
        clear_output(wait=True)
        r, rs = STATE["rfm"], STATE["res"]; cid = w_id.value
        if cid not in r.index:
            print(f"CustomerID {cid} not found in the current selection ({STATE['country']}). Example IDs: {list(r.index[:5])}"); return
        row = r.loc[cid]; seg = rs["seg"].loc[cid]
        display(HTML(f"<h3 style='margin:6px 0'>Customer {cid} · <span style='color:{SEG_COLORS.get(seg, '#333')}'>{seg}</span></h3>" +
                     card("Recency", f"{row.Recency:.0f} days") + card("Frequency", f"{row.Frequency:.0f} invoices") + card("Monetary", f"£{row.Monetary:,.0f}")))
        print("Recommended action:", ACTIONS[seg])
        pct = (r[RFM_COLS] <= row[RFM_COLS]).mean() * 100
        print(f"Percentile within selection: Recency {pct.Recency:.0f}th (lower is better), Frequency {pct.Frequency:.0f}th, Monetary {pct.Monetary:.0f}th")
        hist = df[df.CustomerID == cid]
        print(f"Purchase history: {hist.InvoiceNo.nunique()} invoices, {hist.InvoiceDate.min():%d %b %Y} to {hist.InvoiceDate.max():%d %b %Y}, {hist.Country.iat[0]}")
        fig, ax = plt.subplots(figsize=(8, 3))
        m = hist.set_index("InvoiceDate").TotalAmount.resample("MS").sum()
        ax.bar(m.index, m.values, width=20, color="#0072B2"); ax.set_title("Monthly spend of this customer (£)"); ax.tick_params(axis="x", rotation=30)
        plt.tight_layout(); plt.show()

def rand_customer(_):
    w_id.value = int(STATE["rfm"].sample(1).index[0]); lookup()
w_find.on_click(lookup); w_rand.on_click(rand_customer)
tab2 = W.VBox([W.HBox([w_id, w_find, w_rand]), out_find])

# ---- Tab 3: what-if ----
w_r = W.IntSlider(value=30, min=0, max=int(rfm.Recency.max()), description="Recency (days)", continuous_update=False, layout=L(width="460px"))
w_f = W.IntSlider(value=5, min=1, max=60, description="Frequency", continuous_update=False, layout=L(width="460px"))
w_m = W.FloatLogSlider(value=1500, base=10, min=1, max=5.3, step=0.05, description="Monetary (£)", continuous_update=False, layout=L(width="460px"))
out_wi = W.Output()

def whatif(_=None):
    with out_wi:
        clear_output(wait=True)
        if "res" not in STATE: print("Run the segmentation first."); return
        seg, runner, ratio, dist = assign([w_r.value, w_f.value, w_m.value], STATE["res"])
        conf = "clear fit" if ratio >= 1.5 else ("moderate fit" if ratio >= 1.15 else "borderline: small changes in behaviour could change the segment")
        display(HTML(f"<h3 style='margin:6px 0'>Assigned segment: <span style='color:{SEG_COLORS.get(seg, '#333')}'>{seg}</span></h3>" +
                     card("Runner-up", runner) + card("Distance ratio", f"{ratio:.2f}") + card("Confidence", conf.split(':')[0])))
        print("Recommended action:", ACTIONS[seg]); print(f"Distance to centre {dist:.2f}; runner-up is {ratio:.2f}x farther ({conf}).")
        r = STATE["rfm"]; rs = STATE["res"]
        fig, axs = plt.subplots(1, 3, figsize=(14, 3.2))
        for ax, c, v in zip(axs, RFM_COLS, [w_r.value, w_f.value, w_m.value]):
            ax.hist(np.log1p(r[c]), bins=40, color="#C9D1D9"); ax.axvline(np.log1p(v), color="#D55E00", lw=2)
            ax.set_title(f"{c}: your value vs customers (log scale)"); ax.set_yticks([])
        plt.tight_layout(); plt.show()
for w in (w_r, w_f, w_m): w.observe(whatif, names="value")
tab3 = W.VBox([W.HTML("Adjust a hypothetical customer. Uses the segmentation currently selected in tab 1."), w_r, w_f, w_m, out_wi])

# ---- Tab 4: export ----
w_exp = W.Button(description="Export results (CSV + ZIP)", icon="download", button_style="success", layout=L(width="260px"))
out_exp = W.Output()
def export(_=None):
    with out_exp:
        clear_output(wait=True)
        r, rs = STATE["rfm"], STATE["res"]; tag = f"{STATE['country'].replace(' ', '_')}_{rs['algo'].replace(' ', '_')}_k{rs['k']}"
        files_out = []
        for name, frame in {f"customer_segments_{tag}.csv": r.join(rs["seg"]),
                            f"segment_profiles_{tag}.csv": segment_profile(r, rs)}.items():
            p = os.path.join(RESULTS_DIR, name); frame.to_csv(p); files_out.append(p)
        zp = f"Customer_Segmentation_Results_{tag}.zip"
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            for p in files_out: z.write(p, os.path.basename(p))
        print("Saved:", *files_out, zp, sep="\n  ")
        try:
            from google.colab import files as colab_files; colab_files.download(zp)
        except ImportError:
            print("(Not running in Colab: files stay on disk.)")
w_exp.on_click(export)
tab4 = W.VBox([W.HTML("Exports the customer-level segment file and profile table for the current selection."), w_exp, out_exp])

tabs = W.Tab(children=[tab1, tab2, tab3, tab4])
for i, t in enumerate(["Segmentation", "Customer lookup", "What-if", "Export"]): tabs.set_title(i, t)
display(HTML("<h2 style='margin:4px 0'>Customer Segmentation Dashboard</h2>"))
display(tabs)
run_segmentation()
lookup(); whatif()

## 11. Save outputs for the report
Writes the standard result files to `Results/` and a ZIP you can attach to your CA2 submission.

In [ ]:
rfm_seg = rfm.join(res["seg"]).join(rfm_scored[["R", "F", "M", "RFM_Score", "RuleSegment"]])
rfm_seg.to_csv(f"{RESULTS_DIR}/customer_segments.csv")
profile.to_csv(f"{RESULTS_DIR}/segment_profiles.csv")
ksel.to_csv(f"{RESULTS_DIR}/k_selection.csv")
comp.to_csv(f"{RESULTS_DIR}/algorithm_comparison.csv")
audit_df.to_csv(f"{RESULTS_DIR}/data_audit.csv"); flow.to_csv(f"{RESULTS_DIR}/cleaning_flow.csv", index=False)

with zipfile.ZipFile("Customer_Segmentation_CA2_Results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(RESULTS_DIR)):
        z.write(os.path.join(RESULTS_DIR, f), f)
print("Saved:", sorted(os.listdir(RESULTS_DIR)))
try:
    from google.colab import files; files.download("Customer_Segmentation_CA2_Results.zip")
except ImportError:
    pass